# Global Topic Renaming — iGEM Teams (Part 2)

The per-cluster naming in Part 1 works locally: each topic is named in
isolation. This can produce **duplicate or ambiguous names** when two clusters
cover related sub-themes (e.g. both named "History of Synthetic Biology").

This notebook fixes that by giving the LLM a **global view** of all
**iGEM Teams** topics at once. We use **OpenAI function calling** so the
model returns a structured array of `(topic_id, name)` pairs — one per cluster —
guaranteeing distinct, publication-ready names.

Overwrites `teams_topic_names.txt`, adding a `global_name` column.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 03-topic_names/, where the aux/ package
# resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from aux.paths import MODELS_DIR, OPENAI_MODEL
from aux.openai_client import load_prompts, make_client
from aux.tables import load_topic_names, save_topic_names
from aux.global_rename import rename_topics_global

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
PREFIX = "teams"
MODEL  = OPENAI_MODEL

prompts = load_prompts()
client = make_client()

## 1. Load Part 1 results

In [3]:
names = load_topic_names(PREFIX)
print(f"Teams: {len(names)} topics")
names[["topic", "name", "description"]].head()

Teams: 161 topics


,topic,name,description
0,0,Synthetic Biology for Spatial and Controlled G...,This cluster centers on advanced strategies fo...
1,1,Plastic Biodegradation and Recycling,This cluster centers on leveraging synthetic b...
2,2,Synthetic Biology for Environmental Monitoring...,This cluster centers on the innovative use of ...
3,3,Bacterial Cancer Synthetic Biology,The overarching theme centers on harnessing sy...
4,4,Synthetic Biology in Diagnostics and Forensics,This cluster focuses on leveraging advanced bi...


## 2. Global rename (function calling)

In [4]:
renamed = rename_topics_global(names, client, prompts, model=MODEL)
renamed[["topic", "name", "global_name", "description"]]

,topic,name,global_name,description
0,0,Synthetic Biology for Spatial and Controlled G...,Spatial and Stimuli-Responsive Gene Regulation,This cluster centers on advanced strategies fo...
1,1,Plastic Biodegradation and Recycling,Plastic Biodegradation and Recycling,This cluster centers on leveraging synthetic b...
2,2,Synthetic Biology for Environmental Monitoring...,Environmental Monitoring and Bioremediation,This cluster centers on the innovative use of ...
3,3,Bacterial Cancer Synthetic Biology,Bacterial Cancer Diagnostics and Therapy,The overarching theme centers on harnessing sy...
4,4,Synthetic Biology in Diagnostics and Forensics,Synthetic Diagnostics and Forensics,This cluster focuses on leveraging advanced bi...
...,...,...,...,...
156,156,Synthetic Biology for Blood Safety and Compati...,Blood Compatibility and Safety,This cluster focuses on harnessing synthetic b...
157,157,Heavy Metal Synthetic Biology,Heavy Metal Detection and Detoxification,This cluster focuses on advancing synthetic bi...
158,158,Synthetic Biology for Sustainable Textile Indu...,Sustainable Textile Manufacturing,This cluster focuses on leveraging synthetic b...
159,159,Synthetic Biology for Noninvasive Oral Cancer ...,Noninvasive Oral Cancer Diagnostics,This cluster focuses on harnessing synthetic b...


## 3. Save final results

In [5]:
save_topic_names(renamed, PREFIX)
print(f"Saved → {MODELS_DIR / f'{PREFIX}_topic_names.txt'} (added global_name)")

Saved → /Users/cristian/Desktop/GitHub/igem-synbio/assets/topic_models/teams_topic_names.txt (added global_name)
